# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = None # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

In [ ]:
#Подключение функций
from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ


| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Еженедельная подборка увеличит средний оборот в партнерских сервисах за четыре недели минимум на 3% относительно контроля. Для планирования 3% от среднего предпериода - 94,99 ₽ на клиента. |
| единица рандомизации и почему именно она (подпиской делятся - колонка `members`) | Подписка: владелец и все ее участники должны попасть в одну группу, иначе скидкой могут воспользоваться люди из контроля. Единица анализа в выданной таблице - клиент. Здесь нет идентификатора общей подписки, поэтому расчеты задания предполагают независимость строк; проверить это предположение по `members` нельзя. Если несколько строк относятся к одной подписке, нужно учитывать группы связанных клиентов и заново считать неопределенность и размер выборки. |
| группы и доли | Тест получает одну подборку в неделю, контроль - обычные коммуникации без этой подборки. План: 6 000 клиентов, 50/50, по 3 000 в каждой группе. Сравниваю всех назначенных в группы, независимо от открытия push. |
| основная метрика | `spend`: средний оборот в партнерских сервисах на клиента за четыре недели, включая клиентов без покупок. Это оборот, а не выручка ПРАЙМ. `purchases` - дополнительная метрика. |
| метрики-стражи: что не должно ухудшиться | `churned` и `tickets`. Для решения принимаю допустимые пределы роста: 0,5 процентного пункта оттока и 0,02 обращения на клиента. Верхние границы 95% интервалов должны быть ниже этих пределов; одного незначимого p недостаточно. |
| α, мощность, MDE - числами из ячейки выше | Двусторонний тест, α = 0,05, мощность 80%. По предпериоду σ = 3 286,86 ₽ и ρ = 0,601. `mde_for_n` дает 237,76 ₽ без CUPED и 189,99 ₽ с CUPED, то есть 7,51% и 6,00% от среднего предпериода. Поэтому 3 000 клиентов на группу недостаточно для надежного обнаружения роста на 3%. |
| поправка на множественные сравнения | Для основной заранее выбранной метрики - одно сравнение при α = 0,05. При одновременной проверке четырех метрик применяю поправку Холма. Уточнение CUPED и остальные методы служат проверкой результата, а не поиском самого маленького p. |
| оценщик | Основная оценка - разность средних теста и контроля после CUPED с `spend_prev` и ее 95% интервал. Дополнительно считаю обычную разность, бутстреп и мэтчинг. Ковариата измерена до отправки push. |
| окно | Четыре полные недели, по одной подборке каждую неделю; `spend_prev` - предыдущие четыре недели. Проверяю результат после окончания окна, без остановки при первом значимом p. Для планирования CUPED использую связь двух предпериодов. |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Расширяем запуск, если разбиение не вызывает сомнений, нижняя граница интервала основной оценки выше нуля, точечный рост не меньше 3%, стражи укладываются в пределы и денежный эффект после всех затрат положителен. Если эффект перспективный, но полные затраты или корректность разбиения еще не проверены, сначала уточняем их и расширяем поэтапно с контролем. Если подтвержден вред стражам или отрицательная экономика, отказываемся от текущей версии. |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test['spend'], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
print("Уникальных клиентов:", mine["client_id"].nunique())
print("Повторов client_id:", mine["client_id"].duplicated().sum())
check_columns = ["spend", "spend_prev", "spend_prev2", "purchases", "churned", "tickets"]
print("Пропусков в основных расчетах:", mine[check_columns].isna().sum().sum())
print(f"SRM p = {srm.pvalue:.3f}, баланс p = {balance.pvalue:.3f}")
print(f"Эффект {effect['diff']:.2f} ₽, плановый MDE {mde_raw:.2f} ₽")
print(f"Ноль входит в интервал: {effect['lo'] <= 0 <= effect['hi']}")
print(f"HypEx t-test p = {answer_4[0]:.4f}, U-test p = {answer_4[1]:.4f}")


#### ✍️ Ответ на вопрос 1
В данных 6 000 уникальных клиентов, а p SRM = 0,624 и p проверки предпериода = 0,919: заметного нарушения пропорции и баланса не обнаружено, хотя это не доказывает отсутствие всех ошибок рандомизации. Обычная оценка эффекта 240,50 ₽ немного выше планового MDE 237,76 ₽, а интервал [59,85; 421,15] ₽ не включает ноль. В HypEx оба критерия дают значимость: p t-критерия = 0,0091, p U-критерия = 0,0147. Выводы здесь совпадают; вообще они могут различаться, потому что t-критерий сравнивает средние, а U-критерий использует ранги и иначе реагирует на большие значения оборота.


## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
y, x = mine["spend"], mine["spend_prev"]
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
width_raw = effect["hi"] - effect["lo"]
width_cuped = effect_cuped["hi"] - effect_cuped["lo"]
width_reduction = 100 * (1 - width_cuped / width_raw)
prev_diff = test["spend_prev"].mean() - control["spend_prev"].mean()

print(f"Ширина интервала: {width_raw:.2f} → {width_cuped:.2f} ₽, уже на {width_reduction:.2f}%")
print(f"Разница предпериода: {prev_diff:.2f} ₽, поправка: {theta * prev_diff:.2f} ₽")
print("Метрики после Холма:", answer_6)
print(f"Отток: ДИ [{100 * guards.loc['churned', 'lo']:.2f}; {100 * guards.loc['churned', 'hi']:.2f}] п.п.")
print(f"Обращения: ДИ [{guards.loc['tickets', 'lo']:.4f}; {guards.loc['tickets', 'hi']:.4f}]")

#### ✍️ Ответ на вопрос 2

Стражи не показали значимого ухудшения: отток изменился на -0,63 п.п. при p χ² = 0,195, обращения - на -0,0067 на клиента при p = 0,250; их верхние границы интервалов +0,27 п.п. и +0,0047 укладываются в выбранные пределы 0,5 п.п. и 0,02 обращения. После поправки Холма значимы только `spend` и `purchases`, а снижение оттока и обращений не доказано. CUPED уменьшил дисперсию на 41,8%, но ширину интервала - на 23,76%, с 361,30 до 275,44 ₽, и немного изменил эффект: с 240,50 до 234,42 ₽. В предпериоде тест тратил в среднем 3 170,68 ₽, контроль - 3 162,01 ₽, поэтому CUPED вычел из разности 0,702 × 8,67 = 6,09 ₽ исходного преимущества теста.

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
rng = np.random.default_rng(VARIANT)
x_test = test['spend'].to_numpy()
x_control = control['spend'].to_numpy()
boostrap_diffs= []

for i in range(2000):
    sample_test = rng.choice(x_test, size=len(x_test), replace=True)
    sample_control = rng.choice(x_control, size=len(x_control), replace=True)
    boostrap_diffs.append(sample_test.mean() - sample_control.mean())

boostrap_ci = np.percentile(boostrap_diffs, [2.5, 97.5])

answer_9 = [round(float(value), 2) for value in boostrap_ci]
answer_9

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
candidates = mine[['spend_prev', 'spend_prev2', 'purchases_prev', 'sessions_prev', 'sessions_prev2']].copy()
candidates['spend_prev+spend_prev2'] = mine['spend_prev'] + mine['spend_prev2']

rho_squared = candidates.corrwith(mine['spend']) ** 2 * 100
rho_squared = rho_squared.sort_values(ascending=False)
purchases_rho_squared = mine['purchases'].corr(mine['spend']) ** 2 * 100

best_covariate = str(rho_squared.idxmax())

answer_10 = [best_covariate, round(float(rho_squared.loc[best_covariate]), 1)]
answer_10

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
features = ['age', 'tenure_days', 'kyc_level', 'members', 'spend_prev', 'sessions_prev']
matching_data = mine.copy()
matching_data['t'] = (matching_data['grp'] == 'test').astype(int)
matching_roles = {'client_id': InfoRole(str), 't': TreatmentRole(), 'spend': TargetRole(), **{name: FeatureRole() for name in features}}

matching_result = Matching(n_neighbors=5).execute(Dataset(roles=matching_roles, data=matching_data, default_role=InfoRole()))
matching_table = F(matching_result.resume)

ate = matching_table.loc['ATE']
answer_11 = [round(float(ate[name]), 2) for name in ('Effect Size', 'CI Lower', 'CI Upper')]
answer_11

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽
commission_gain = effect_cuped['diff'] * COMMISSION
ticket_cost_change = guards.loc['tickets', 'diff'] * TICKET_COST
churn_cost_change = guards.loc['churned', 'diff'] * CHURN_COST
money_effect = commission_gain - ticket_cost_change - churn_cost_change

money = COMMISSION * mine["spend_cuped"] - TICKET_COST * mine["tickets"] - CHURN_COST * mine["churned"]
money_ci = diff_ci(money[mine["grp"] == "test"], money[mine["grp"] == "control"])
answer_12 = round(float(money_effect), 2)
answer_12


In [ ]:
print("Комиссия:", round(commission_gain, 2))
print("Изменение расходов на поддержку:", round(ticket_cost_change, 2))
print("Изменение потерь от оттока:", round(churn_cost_change, 2))
print("Итого:", answer_12)

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
widths = pd.Series({
    "Обычный": effect["hi"] - effect["lo"],
    "CUPED": effect_cuped["hi"] - effect_cuped["lo"],
    "Бутстреп": boostrap_ci[1] - boostrap_ci[0],
    "Мэтчинг": float(ate["CI Upper"] - ate["CI Lower"])
})
print("Ширина 95% интервалов, ₽:")
print(widths.round(2).to_string())
print(f"Рост оборота после CUPED: {100 * effect_cuped['diff'] / control['spend'].mean():.2f}%")
print(f"ρ² для purchases: {purchases_rho_squared:.2f}%")
print(f"Денежный эффект: {money_effect:.2f} ₽, ДИ [{money_ci['lo']:.2f}; {money_ci['hi']:.2f}] ₽")

#### ✍️ Ответ на вопрос 3

Я больше доверяю заранее выбранному CUPED с `spend_prev`: в рандомизированном эксперименте он использует данные до акции и дает оценку 234,42 ₽ с интервалом шириной 275,44 ₽ против 361,30 ₽ у обычной оценки, 346,32 ₽ у бутстрепа и 287,14 ₽ у мэтчинга. CUPED убирает часть различий между клиентами, бутстреп учитывает форму наблюдаемого распределения, но сам по себе не убирает этот шум, а мэтчинг требует дополнительных допущений и здесь не нужен для устранения отбора. У `purchases` ρ² = 69,30%, но покупки измерены во время эксперимента и зависят от акции: такая поправка может убрать часть самого эффекта; среди допустимых признаков лучшая сумма двух прошлых оборотов с ρ² = 46,29%, однако менять основной оценщик после просмотра результатов я не буду. В принятой модели акция дает +9,21 ₽ на клиента с интервалом [1,94; 16,48] ₽, но для полной окупаемости не хватает затрат на подготовку подборок, отправку push, разработку, финансирование скидок и оценки результата после четырех недель.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:**  четыре недели отправляли подборку партнерских скидок один раз в неделю: в тесте 2 981 клиент, в контроле 3 019. Существенного перекоса размеров групп не обнаружено, p SRM = 0,624; средний оборот до акции также был сопоставим, p = 0,919.

**Что получилось:** оценка прироста оборота после CUPED - 234,42 ₽ на клиента, или 7,09% от среднего контроля, с 95% интервалом [96,70; 372,14] ₽. Отток изменился на -0,63 п.п., обращения - на -0,0067 на клиента; подтвержденного ухудшения нет, а верхние границы их интервалов +0,27 п.п. и +0,0047 ниже выбранных пределов 0,5 п.п. и 0,02.

**Какие остались вопросы:** при комиссии 1,5%, цене обращения 150 ₽ и потере от ушедшего клиента 750 ₽ расчет дает +9,21 ₽ на клиента, интервал [1,94; 16,48] ₽. Это результат до затрат на саму акцию, а снижение оттока и обращений по отдельности не доказано; также нужно проверить, не оказались ли участники одной подписки в разных группах, и учесть, что наблюдение длилось только четыре недели.

**Что рекомендуем:** на всех сразу пока не запускать; подготовить поэтапное расширение с сохранением контроля. Пилот показывает положительный эффект, но сначала нужно подтвердить корректность разделения общих подписок и стоимость запуска: дополнительные расходы выше 9,21 ₽ на клиента съедят точечный выигрыш, а для осторожного решения стоит ориентироваться на нижнюю границу 1,94 ₽ за четыре недели. После этой проверки расширять аудиторию, продолжая следить за оттоком, поддержкой и полной экономикой.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А
median_sessions = mine["sessions_prev"].median()
active_test = mine[(mine["grp"] == "test") & (mine["sessions_prev"] >= median_sessions)]
active_control = mine[(mine["grp"] == "control") & (mine["sessions_prev"] >= median_sessions)]

observational = mine[
    (mine["grp"] == "control") | (mine["sessions_prev"] >= median_sessions)
].copy().reset_index(drop=True)
observational["t"] = (observational["grp"] == "test").astype(int)

naive = diff_ci(active_test["spend"], control["spend"])
benchmark = diff_ci(active_test["spend"], active_control["spend"])

roles_a = {
    "client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
    **{name: FeatureRole() for name in features}
}
result_a = Matching(n_neighbors=5).execute(
    Dataset(roles=roles_a, data=observational, default_role=InfoRole())
)
att_full = F(result_a.resume).loc["ATT"]

features_short = [name for name in features if name != "sessions_prev"]
roles_short = {
    "client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
    **{name: FeatureRole() for name in features_short}
}
result_short = Matching(n_neighbors=5).execute(
    Dataset(roles=roles_short, data=observational, default_role=InfoRole())
)
att_short = F(result_short.resume).loc["ATT"]

comparison_a = pd.DataFrame([
    {"method": "Наивная разность", "effect": naive["diff"], "lo": naive["lo"], "hi": naive["hi"]},
    {"method": "Эталон: активные в двух группах", "effect": benchmark["diff"], "lo": benchmark["lo"], "hi": benchmark["hi"]},
    {"method": "ATT по шести признакам", "effect": att_full["Effect Size"], "lo": att_full["CI Lower"], "hi": att_full["CI Upper"]},
    {"method": "ATT без sessions_prev", "effect": att_short["Effect Size"], "lo": att_short["CI Lower"], "hi": att_short["CI Upper"]}
])
comparison_a["difference_from_benchmark"] = comparison_a["effect"] - benchmark["diff"]

print(f"Медиана сессий: {median_sessions:.0f}")
print(f"Активных: тест {len(active_test)}, контроль {len(active_control)}")
print(f"Наблюдательная выборка: тест {len(active_test)}, контроль {len(control)}")
print(comparison_a.round(2).to_string(index=False))
print("Проверки баланса HypEx:")
print(F(result_a.quality_results).to_string())
print("Диапазон сессий активного теста:", active_test["sessions_prev"].min(), active_test["sessions_prev"].max())
print("Диапазон сессий контроля:", control["sessions_prev"].min(), control["sessions_prev"].max())

#### ✍️ Исследование А
**Вопрос.** Насколько отбор только активных клиентов искажает оценку акции и помогает ли мэтчинг?

**Как проверяла.** Порог активности - медиана `sessions_prev`, равная 15: оставила 1 491 активного клиента теста и всех 3 019 клиентов контроля. Посчитала наивную разность и ATT с пятью соседями по шести признакам, затем без `sessions_prev`; эталон - сравнение активного теста с 1 572 активными клиентами контроля, где сохраняется исходная рандомизация.

**Что получилось.** Наивная разность - 1 781,99 ₽, а экспериментальный эталон среди активных - 402,14 ₽ с интервалом [102,38; 701,89] ₽: наивная оценка выше на 1 379,85 ₽. Полный мэтчинг дает ATT 312,98 ₽ с интервалом [85,78; 540,18] ₽, то есть точечную оценку на 89,16 ₽ ниже эталона. Без `sessions_prev` ATT возрастает до 726,24 ₽ с интервалом [492,16; 960,32] ₽ и оказывается выше эталона на 324,10 ₽: мы убрали сам признак отбора, связанный с оборотом, и остальные признаки не полностью его заменили.

**Что это значит.** Учет активности заметно приблизил оценку к эталону, но проверки HypEx не показывают идеального баланса по всем признакам, а максимальное число сессий в активном тесте - 198 против 176 в контроле, поэтому для части клиентов близких аналогов мало. Для оценки эффекта на тех, кому показали акцию, нужен ATT, а сравнение с общим ATE отвечает на другой вопрос. Мэтчингу можно доверять, если измерены важные причины отбора и оборота, есть сопоставимые клиенты в контроле и после подбора достигнут баланс. Близость к эталону здесь не доказывает отсутствие скрытых причин отбора в реальном наблюдательном исследовании; сам эталон тоже оценен с неопределенностью.

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б
# Коэффициент и скорректированную метрику считаем только по контролю.
control_cuped_b, theta_b = cuped(control["spend"], control["spend_prev"])
delta_b = 0.03 * control["spend"].mean()
n_formula_raw = n_per_group(control["spend"].std(), delta_b)
n_formula_cuped = n_per_group(control_cuped_b.std(), delta_b)
n_grid = sorted(set([
    1000, 3000, 10000, 12000, 20000, 40000,
    n_formula_raw, n_formula_cuped
]))

rng = np.random.default_rng(VARIANT)
power_rows = []

for n in n_grid:
    row = {"n": n}
    for name, values in [("raw", control["spend"]), ("cuped", control_cuped_b)]:
        y = values.to_numpy()
        rejected = 0

        for _ in range(500):
            sample = rng.choice(y, size=2 * n, replace=True)
            p = stats.ttest_ind(sample[:n] + delta_b, sample[n:], equal_var=False).pvalue
            rejected += p < 0.05

        row[name + "_simulation"] = rejected / 500
        z_effect = delta_b / (values.std() * np.sqrt(2 / n))
        # Приближение мощности из семинара С7.
        row[name + "_formula"] = float(stats.norm.cdf(z_effect - stats.norm.ppf(0.975)))

    power_rows.append(row)

power_table = pd.DataFrame(power_rows)
print(f"Эффект +3%: {delta_b:.2f} ₽; θ по контролю: {theta_b:.3f}")
print(f"По формуле без CUPED нужно {n_formula_raw} на группу")
print(f"По формуле с CUPED нужно {n_formula_cuped} на группу")
print(power_table.round(4).to_string(index=False))

for method in ["raw", "cuped"]:
    enough = power_table.loc[power_table[method + "_simulation"] >= 0.8, "n"]
    first_n = int(enough.iloc[0]) if len(enough) > 0 else None
    print(f"{method}: первое n с мощностью от 80% на сетке - {first_n}")

mc_error = 1.96 * np.sqrt(0.8 * 0.2 / 500)
print(f"Погрешность при мощности около 80%: ±{100 * mc_error:.1f} п.п.")

#### ✍️ Исследование Б

**Вопрос.** Достаточно ли 3 000 клиентов на группу, чтобы заметить рост оборота на 3%, и согласуется ли формула мощности с симуляцией?

**Как проверяла.** Взяла оборот контроля и добавляла тестовой половине 99,19 ₽ - 3% от среднего контроля. Для каждого размера сделала 500 выборок с возвращением и применяла двусторонний критерий Уэлча при α = 0,05; затем повторила расчет на готовой CUPED-метрике, посчитанной только по контролю с θ = 0,665. Оборот и прошлый оборот по отдельности не перемешивала, а формулу взяла из семинара С7.

**Что получилось.** При 3 000 на группу мощность - 20,0% без CUPED и 32,8% с CUPED; формула дает 21,0% и 31,7%. Для мощности 80% она требует 17 698 и 10 689 клиентов на группу соответственно. В симуляции первые значения от 80% на проверенной сетке - 20 000 без CUPED и 10 000 с CUPED, но это не точные минимумы: с CUPED при 10 000 получилось 80,4%, при 10 689 - 78,6%, а при 12 000 - 81,6%. При 500 повторах погрешность около уровня 80% составляет примерно ±3,5 п.п., поэтому такие колебания ожидаемы.

**Что это значит.** Пилот на 6 000 человек недостаточно мощный для небольшого роста на 3%, хотя более крупный эффект он обнаружить может. Для планирования ориентиры здесь - около 17,7 тысячи на группу без CUPED и 10,7 тысячи с ним; числа отличаются от планирования по предпериоду, потому что в этом исследовании использованы оборот и связь метрик только в контроле. В симуляции нет явного большого расхождения с формулой из-за тяжелого хвоста, но ресэмплинг 3 019 клиентов не создает редкие экстремальные значения, которых не было в выборке, и не гарантирует такую же мощность для другой аудитории или другого вида эффекта.